In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
df = pd.read_csv("movies.txt")  
df.head()
CURRENT_YEAR = 2026
def assign_box_office_bin(box_office):
    if box_office < 100_000_000:
        return 0
    elif box_office < 300_000_000:
        return 1
    elif box_office < 600_000_000:
        return 2
    elif box_office < 1_000_000_000:
        return 3
    else:
        return 4

df["box_office_bin"] = df["box_office_worldwide"].apply(assign_box_office_bin)
y = df["box_office_bin"]
X = df.drop(columns=["box_office_bin", "box_office_worldwide", "title"])
categorical_features = ["distributor_tier"]

numerical_features = [
    "release_year",
    "franchise",
    "franchise_size",
    "budget_tier",
    "sequel",
    "director_brand",
    "family_appeal",
    "is_action",
    "is_scifi",
    "is_superhero",
    "is_comedy",
    "is_animation",
    "is_drama",
    "is_horror"
]
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)
model = RandomForestClassifier(
    n_estimators=1000,
    max_depth=12,
    min_samples_leaf=2,
    min_samples_split=5,
    class_weight="balanced",
    random_state=42
)
pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("classifier", model)
    ]
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))
import numpy as np

mae = np.mean(np.abs(y_test - y_pred))
print("Mean absolute bin error:", mae)

import pandas as pd
import matplotlib.pyplot as plt

preprocessor = pipeline.named_steps["preprocessing"]

feature_names = preprocessor.get_feature_names_out()

importances = pipeline.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

importance_df = importance_df.sort_values("importance", ascending=False)

print(importance_df)

plt.figure()
plt.bar(importance_df["feature"], importance_df["importance"])
plt.xticks(rotation=90)
plt.title("Feature Importance")
plt.show()

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax)

plt.title("Model Performance for test set: Confusion Matrix")
plt.xlabel("Predicted Box Office Bin")
plt.ylabel("Actual Box Office Bin")

plt.show()
import matplotlib.pyplot as plt
bin_counts = df["box_office_bin"].value_counts().sort_index()
plt.figure(figsize=(8, 6))
plt.bar(bin_counts.index, bin_counts.values)
plt.title("Distribution of Films Across Box Office Revenue Bins")
plt.xlabel("Box Office Revenue Bin")
plt.ylabel("Number of Films")
plt.xticks(
    ticks=[0,1,2,3,4],
    labels=[
        "< $100M",
        "$100M–$300M",
        "$300M–$600M",
        "$600M–$1B",
        "> $1B"
    ]
)
plt.show()